In [1]:
import sys
import argparse

from datetime import datetime
import os
import scripts.utils_forTraining as utils
import pandas as pd
import numpy as np

from EPInformer.models import EPInformer_v2, enhancer_predictor_256bp
from scipy import stats
from tqdm import tqdm
import torch
from torch.utils.data import Subset, Dataset
from sklearn.model_selection import GroupKFold

In [4]:
def generate_splits(df, group_name, n_folds=12, seed = 42):
    np.random.seed(seed)
    #df = df.reset_index()
    groups = df[group_name]
    chrs = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr11', 'chr12', 'chr8',  'chr10', 'chr17', 'chr9', 'chr16', 'chr19', 'chr15', 'chrX', 'chr20', 'chr13', 'chr14', 'chr18', 'chr21', 'chr22']
    folds = []

    for i in range(0, n_folds):
        # test_groups
        if (23-i >= len(chrs)):
          test_groups = [chrs[i]]
        else:
          test_groups = [chrs[i], chrs[23-i]]
        # valid_groups
        if i >= 11:
          val_groups = [chrs[0]]
        else:
          val_groups = [chrs[i+1], chrs[22-i]]
        print(test_groups)
        print(val_groups)

        test_idx = np.where(np.isin(groups, test_groups))[0]
        val_idx = np.where(np.isin(groups, val_groups))[0]
        train_idx = np.where(~np.isin(groups, test_groups + val_groups))[0]

        folds.append({
            'train_idx': df.loc[train_idx, 'Ensembl_ID'],
            'val_idx': df.loc[val_idx, 'Ensembl_ID'],
            'test_idx': df.loc[test_idx, 'Ensembl_ID']
        })

    print(f"Generated {len(folds)} round-robin paired folds.")
    return folds

In [6]:
def print_splits(df, folds):
 
    new_split_df = df.copy()    
    new_split_df = new_split_df.set_index('Ensembl_ID')
    new_split_df = new_split_df.drop(columns=new_split_df.columns)
    for fi in range(1, 13):
        train_ensid = folds[int(fi)-1]['train_idx'].tolist()
        valid_ensid = folds[int(fi)-1]['val_idx'].tolist()
        test_ensid = folds[int(fi)-1]['test_idx'].tolist()
        colname = 'fold'+str(fi)
        new_split_df[colname] = ''
        new_split_df.loc[train_ensid,colname] = 'train'
        new_split_df.loc[valid_ensid,colname] = 'valid'
        new_split_df.loc[test_ensid,colname] = 'test'
    print(new_split_df)
    new_split_df.to_csv("split.txt")

In [7]:
cell = 'GM12878'
distance_threshold = 100_000
n_epoch = 100
hic_threshold = None
n_extraFeat = 3
use_pretrained = False
rna_encoding = True
fold_list = 'all'
n_encoder = 3
batch_size = 16
expr_type = 'CAGE'
n_enhancers = 60

EP_df = pd.read_csv('./data/' + 'K562_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
promoter_df = EP_df.groupby('TargetGeneEnsembl_ID', as_index = False)['chr'].first()
promoter_df.rename(columns={'TargetGeneEnsembl_ID': 'Ensembl_ID'}, inplace=True)
all_ds = utils.promoter_enhancer_dataset(data_folder= './data/', expr_type=expr_type, cell_type=cell, n_extraFeat=n_extraFeat, usePromoterSignal=True, n_enhancers=n_enhancers, hic_threshold=hic_threshold, distance_threshold=distance_threshold, rna_encoding=rna_encoding)
ensid_list = [eid.decode() for eid in all_ds.data_h5['ensid'][:]]
ensid_df = pd.DataFrame(ensid_list, columns=['ensid'])
ensid_df['idx'] = np.arange(len(ensid_list))
ensid_df = ensid_df.set_index('ensid')

rna_enc = 'rna_encoding'

EP_df.to_csv(f'./testing/EP_{rna_enc}_df.tsv', sep='\t', index=False)
promoter_df.to_csv(f'./testing/promoter_{rna_enc}_df.tsv', sep='\t', index=False)
ensid_df.to_csv(f'./testing/ensid_{rna_enc}_df.tsv', sep='\t', index=False)
all_ds.promoter_df.to_csv(f'./testing/all_ds_promoter_{rna_enc}_df.tsv', sep='\t', index=False)
print(f'{all_ds.data_h5["ensid"][0]} with promoter code (length {len(all_ds.data_h5["pe_code"][0][:1].squeeze())}: \n{all_ds.data_h5["pe_code"][0][:1].squeeze()}')
file = open(f'./testing/all_ds_{rna_enc}_.txt', 'w')
print(all_ds[0], file=file)
file.close()

b'ENSG00000310526' with promoter code (length 2000: 
[[ 1.         0.         0.         0.        -0.7022614]
 [ 0.         0.         0.         1.        -0.7022614]
 [ 0.         0.         0.         1.        -0.7022614]
 ...
 [ 0.         0.         0.         1.         0.       ]
 [ 1.         0.         0.         0.         0.       ]
 [ 1.         0.         0.         0.         0.       ]]


In [23]:
import os
import pandas as pd
import numpy as np
from sklearn import metrics
from scipy.stats import pearsonr

In [24]:
os.getcwd()

'/home/witoslaw/diffTSS'

In [28]:
R_sum_gm = 0
MAE_sum_gm = 0
MSE_sum_gm = 0
R_sum_k5 = 0
MAE_sum_k5 = 0
MSE_sum_k5 = 0
for fold in range(1,13):
    data_gm = pd.read_csv(f'trained_models/2025-05-06-15/fold_{fold}_EPInformer-PE-Activity-HiC.5base.64dim.3Trans.4head.TrueBN.TrueLN.TrueFeat.3extraFeat.60enh.TrueRNA_enc.FalseRNA_emb.log10RNA_transform.GM12878.CAGE_predictions.csv')
    R_sum_gm += pearsonr(data_gm['Pred'], data_gm['actual'])[0]
    MAE_sum_gm += metrics.mean_absolute_error(data_gm['Pred'], data_gm['actual'])
    MSE_sum_gm += metrics.mean_squared_error(data_gm['Pred'], data_gm['actual'])
    
    data_k5 = pd.read_csv(f'trained_models/2025-04-17-00/fold_{fold}_EPInformer-PE-Activity-HiC.4base.64dim.3Trans.4head.TrueBN.TrueLN.TrueFeat.3extraFeat.60enh.TrueRNA_enc.K562.CAGE_predictions.csv')
    R_sum_k5 += pearsonr(data_k5['Pred'], data_k5['actual'])[0]
    MAE_sum_k5 += metrics.mean_absolute_error(data_k5['Pred'], data_k5['actual'])
    MSE_sum_k5 += metrics.mean_squared_error(data_k5['Pred'], data_k5['actual'])
print(f"GM12878\n"
      f"R:   {R_sum_gm/12}\n"
      f"MAE: {MAE_sum_gm/12}\n"
      f"MSE: {MSE_sum_gm/12}\n"
      f"K562\n"
      f"R:   {R_sum_k5/12}\n"
      f"MAE: {MAE_sum_k5/12}\n"
      f"MSE: {MSE_sum_k5/12}\n")

GM12878
R:   0.7750396346519158
MAE: 0.5093173327280263
MSE: 0.5149114798399861
K562
R:   0.7170614904202992
MAE: 0.5730430455166574
MSE: 0.6364362342201375



In [ ]:
diffTSS/trained_models/2025-05-06-15/fold_1_EPInformer-PE-Activity-HiC.5base.64dim.3Trans.4head.TrueBN.TrueLN.TrueFeat.3extraFeat.60enh.TrueRNA_enc.FalseRNA_emb.log10RNA_transform.GM12878.CAGE_predictions.csv

In [5]:
pearsonr(data['Pred'], data['actual'])

PearsonRResult(statistic=0.8101420312675875, pvalue=0.0)

In [7]:
metrics.mean_absolute_error(data['Pred'], data['actual'])

0.44880337157079925

In [10]:
metrics.mean_squared_error(data['Pred'], data['actual'])

0.4356615081366686

In [11]:
0.44**2

0.1936